In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# Cell: Model Training and Evaluation (Transformer with Attention)
# =====================================================================
# Performs sequence creation, data splitting, scaling, model training
# (with a Transformer Encoder), hyperparameter tuning (optional),
# evaluation, and saves results into a timestamped subfolder.
# Includes a separate function to reload a model and run predictions.

# --- Standard Library Imports ---
import math
import time
import sys
import os
import logging
from datetime import datetime
from functools import partial # For passing args to Optuna objective
import json

# --- Data Handling and Numerical Computation ---
import numpy as np
import pandas as pd

# --- Machine Learning & Deep Learning ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from optuna.trial import TrialState # For callback check
from optuna.exceptions import DuplicatedStudyError # To handle study deletion attempts

# --- Scikit-learn ---
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score, precision_recall_curve,
    classification_report, confusion_matrix, recall_score # recall_score needed for gmean
)

# --- Imbalanced-learn ---
# NOTE: Undersampling is disabled in this version
try:
    from imblearn.under_sampling import RandomUnderSampler
except ImportError:
    logging.warning("`imbalanced-learn` library not found. Undersampling is disabled anyway.")
    RandomUnderSampler = None

# --- Plotting & Jupyter Integration ---
import matplotlib.pyplot as plt # Still used for final static plot & confusion matrix
import matplotlib.dates as mdates # For formatting dates on plots
import seaborn as sns # Used for confusion matrix
import plotly.graph_objects as go # For live plotting
from plotly.subplots import make_subplots # To create subplots
try:
    from IPython.display import display, Image # To display Plotly widget and saved images in Jupyter
    ipython_display_available = True
except ImportError:
    logging.warning("IPython.display not available. Plots will not be displayed inline.")
    ipython_display_available = False
    # Define dummy functions if display is not available to avoid NameError later
    def display(*args, **kwargs): pass
    def Image(*args, **kwargs): pass
# Note: ipywidgets is used implicitly by FigureWidget

# ========================================================
# Logging Setup
# ========================================================
def setup_logging(verbose=True):
    """Configures the root logger based on the verbosity setting."""
    level = logging.INFO if verbose else logging.WARNING
    log_format = '%(asctime)s - %(levelname)s - %(module)s - %(message)s'
    # Use force=True to allow re-configuration if the script is run multiple times
    logging.basicConfig(level=level, format=log_format, datefmt='%Y-%m-%d %H:%M:%S', force=True)
    logging.info(f"Logging configured to level: {logging.getLevelName(level)}")

# ========================================================
# Helper Functions & Classes (Model, Training, Evaluation)
# ========================================================

# --- Reproducibility Helper ---
def set_seed(seed_value):
    """Sets the random seed for reproducibility across libraries."""
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # Ensure deterministic behavior for CuDNN (can impact performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed_value}")

# --- Sequence Creation and Splitting Helpers ---
def create_sequences(input_data, target_data, seq_length):
    """
    Creates sequences and corresponding labels from time series data.
    **CORRECTED to prevent look-ahead bias.**
    """
    sequences, labels = [], []
    # Ensure numpy arrays
    if isinstance(input_data, pd.DataFrame): input_data = input_data.values
    if isinstance(target_data, pd.Series): target_data = target_data.values

    for i in range(len(input_data) - seq_length):
        # Sequence is features from t-seq_length to t-1
        sequence = input_data[i:i + seq_length]
        # Label is the target at time t
        label = target_data[i + seq_length]
        sequences.append(sequence)
        labels.append(label)

    if not sequences:
        logging.warning(f"Input data length ({len(input_data)}) <= seq length ({seq_length}). Cannot create sequences.")

    return np.array(sequences), np.array(labels)


def log_class_distribution(labels, dataset_name):
    """Logs the class distribution of a label array."""
    if labels is None or len(labels) == 0:
        logging.warning(f"Cannot log class distribution for {dataset_name}: labels are empty or None.")
        return
    try:
        unique_classes, counts = np.unique(labels.astype(int), return_counts=True)
        distribution = dict(zip(unique_classes, counts))
        total_samples = len(labels)
        ratios = {k: f"{(v/total_samples)*100:.2f}%" for k, v in distribution.items()}
        logging.info(f"{dataset_name} class distribution - Counts: {distribution}, Ratios: {ratios}")
    except Exception as e:
        logging.error(f"Could not calculate class distribution for {dataset_name}: {e}")

def split_apply_undersample_scale(data_df, config):
    """
    Splits data chronologically, creates sequences, applies scaling based on the training set,
    and returns the processed data along with the original test indices and scaler.
    """
    logging.info("--- Starting Data Splitting, Sequencing, and Scaling ---")
    set_seed(config['SEED'])

    if 'target' not in data_df.columns: logging.error("Column 'target' not found."); sys.exit(1)
    features_df = data_df.drop('target', axis=1)
    target_series = data_df['target']

    X_raw, y_raw = features_df.values, target_series.values
    original_index = data_df.index
    n_features = X_raw.shape[1]
    logging.info(f"Separated raw features ({n_features}) and target.")

    n_samples_raw = len(X_raw)
    train_end_idx = int(n_samples_raw * config['TRAIN_SPLIT_RATIO'])
    val_end_idx = train_end_idx + int(n_samples_raw * config['VALIDATION_SPLIT_RATIO'])

    X_train_raw, y_train_raw = X_raw[:train_end_idx], y_raw[:train_end_idx]
    X_val_raw, y_val_raw = X_raw[train_end_idx:val_end_idx], y_raw[train_end_idx:val_end_idx]
    X_test_raw, y_test_raw = X_raw[val_end_idx:], y_raw[val_end_idx:]

    logging.info(f"Chronological split: Train={len(X_train_raw)}, Val={len(X_val_raw)}, Test={len(X_test_raw)}")

    logging.info("Creating sequences...")
    seq_length = config['SEQUENCE_LENGTH']
    X_train_seq, y_train_seq = create_sequences(X_train_raw, y_train_raw, seq_length)
    X_val_seq, y_val_seq = create_sequences(X_val_raw, y_val_raw, seq_length)
    X_test_seq, y_test_seq = create_sequences(X_test_raw, y_test_raw, seq_length)

    test_indices_seq = None
    if len(y_test_seq) > 0:
        start_index_for_test_labels = val_end_idx + seq_length
        end_index_for_test_labels = start_index_for_test_labels + len(y_test_seq)
        test_indices_seq = original_index[start_index_for_test_labels:end_index_for_test_labels]
        if len(test_indices_seq) != len(y_test_seq):
            logging.error("Mismatch between test labels and indices length."); test_indices_seq = None
        else:
            logging.info(f"Aligned test indices with sequence labels. Length: {len(test_indices_seq)}")

    if X_train_seq.size == 0 or X_val_seq.size == 0:
        logging.critical("Empty training or validation set after sequencing."); sys.exit(1)

    logging.info(f"Sequences created. Shapes: Train={X_train_seq.shape}, Val={X_val_seq.shape}, Test={X_test_seq.shape}")

    logging.info("Fitting and applying StandardScaler...")
    scaler = StandardScaler()
    train_shape = X_train_seq.shape
    scaler.fit(X_train_seq.reshape(-1, n_features))
    X_train = scaler.transform(X_train_seq.reshape(-1, n_features)).reshape(train_shape)
    X_val = scaler.transform(X_val_seq.reshape(-1, n_features)).reshape(X_val_seq.shape)
    X_test = scaler.transform(X_test_seq.reshape(-1, n_features)).reshape(X_test_seq.shape) if X_test_seq.size > 0 else np.array([])

    logging.info("--- Data Splitting, Sequencing, and Scaling Finished ---")
    return X_train, y_train_seq, X_val, y_val_seq, X_test, y_test_seq, n_features, scaler, y_train_raw, test_indices_seq

# --- PyTorch Dataset ---
class TimeSeriesDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]

# --- Model Architecture Components (Transformer) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerClassifier(nn.Module):
    def __init__(self, n_features: int, d_model: int, n_heads: int, n_encoder_layers: int,
                 dim_feedforward: int, transformer_dropout: float, fc_dropout: float, n_classes: int = 1):
        super().__init__()
        self.d_model = d_model
        self.init_args = locals()
        del self.init_args['self'], self.init_args['__class__']

        self.input_embed = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, transformer_dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=transformer_dropout, batch_first=True, activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_encoder_layers)
        self.dropout = nn.Dropout(fc_dropout)
        self.fc = nn.Linear(d_model, n_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.input_embed.weight); nn.init.zeros_(self.input_embed.bias)
        nn.init.xavier_uniform_(self.fc.weight); nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_embed(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        transformer_out = self.transformer_encoder(x)
        pooled_out = transformer_out.mean(dim=1)
        out = self.dropout(pooled_out)
        logits = self.fc(out)
        return logits.squeeze(-1) if self.init_args['n_classes'] == 1 else logits

# --- Training and Evaluation Functions ---
def train_epoch(model, dataloader, criterion, optimizer, device, grad_clip_norm):
    model.train()
    total_loss, total_grad_norm, batches_processed = 0.0, 0.0, 0
    for sequences, labels in dataloader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        if torch.isnan(loss): continue
        loss.backward()
        total_grad_norm += sum(p.grad.detach().data.norm(2).item() ** 2 for p in model.parameters() if p.grad is not None) ** 0.5
        if grad_clip_norm: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        total_loss += loss.item(); batches_processed += 1
    return (total_loss / batches_processed if batches_processed else 0.0,
            total_grad_norm / batches_processed if batches_processed else 0.0)

def evaluate(model, dataloader, criterion, device, return_preds=False):
    model.eval()
    all_preds_prob, all_labels = [], []
    metrics = {'loss': float('nan'), 'f1': 0.0, 'acc': 0.0, 'auc': 0.5, 'gmean': 0.0}
    if not dataloader: return (metrics, np.array([]), np.array([])) if return_preds else metrics

    with torch.no_grad():
        total_loss = 0.0
        for sequences, labels in dataloader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            if criterion: total_loss += criterion(outputs, labels).item()
            all_preds_prob.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    if criterion: metrics['loss'] = total_loss / len(dataloader)
    all_labels, all_preds_prob = np.array(all_labels), np.array(all_preds_prob)

    if len(all_labels) > 0:
        preds_binary = (all_preds_prob >= 0.5).astype(int)
        metrics.update({
            'f1': f1_score(all_labels, preds_binary, zero_division=0),
            'acc': accuracy_score(all_labels, preds_binary),
            'gmean': math.sqrt(recall_score(all_labels, preds_binary, pos_label=0, zero_division=0) * recall_score(all_labels, preds_binary, pos_label=1, zero_division=0))
        })
        if len(np.unique(all_labels)) > 1:
            try: metrics['auc'] = roc_auc_score(all_labels, all_preds_prob)
            except ValueError: metrics['auc'] = 0.5

    return (metrics, all_labels, all_preds_prob) if return_preds else metrics

# --- Plotting and Summary Functions ---
def save_static_training_plot(history, best_epoch, metric_name, save_path):
    if not history or not history.get('train_loss'): logging.warning("History empty, skipping plot."); return
    fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
    fig.suptitle(f'Training History (Best Val {metric_name.upper()} @ Epoch {best_epoch or "N/A"})', fontsize=16)
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='.'); axes[0].plot(epochs, history['val_loss'], label='Val Loss', marker='.', ls='--'); axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(epochs, history['val_f1'], label='Val F1', marker='.', ls='--'); axes[1].plot(epochs, history['val_acc'], label='Val Acc', marker='.', ls=':'); axes[1].plot(epochs, history['val_auc'], label='Val AUC', marker='.', ls='-.'); axes[1].plot(epochs, history['val_gmean'], label='Val G-mean', marker='^', ls=':'); axes[1].set_title('Metrics'); axes[1].set_ylim(-0.05, 1.05); axes[1].legend()
    axes[2].plot(epochs, history['avg_grad_norm'], label='Avg Grad Norm', marker='.'); axes[2].set_title('Gradient Norm'); axes[2].set_xlabel('Epochs'); axes[2].legend()
    if best_epoch: [ax.axvline(x=best_epoch, color='r', ls=':', lw=2) for ax in axes]
    plt.tight_layout(rect=[0, 0.03, 1, 0.96]); plt.savefig(save_path, dpi=150); plt.close(fig)
    logging.info(f"Static training plot saved: {save_path}")

def plot_confusion_matrix(y_true, y_pred, save_path, title_suffix=""):
    cm = confusion_matrix(y_true, y_pred); plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'])
    plt.title(f'Confusion Matrix{title_suffix}', fontsize=14); plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    logging.info(f"Confusion matrix saved: {save_path}")

def plot_actual_vs_prediction(pred_df, save_path, threshold):
    plt.figure(figsize=(15, 7))
    plt.plot(pred_df.index, pred_df['actual'], label='Actual', marker='o', ls='None', ms=4, alpha=0.7)
    plt.plot(pred_df.index, pred_df['predicted_prob'], label='Probability', alpha=0.8)
    plt.axhline(y=threshold, color='r', ls='--', label=f'Threshold ({threshold:.3f})')
    plt.title('Actual vs. Predicted Probability'); plt.legend(); plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    logging.info(f"Actual vs. prediction plot saved: {save_path}")

def save_results_summary(config, final_params, best_val_metrics, final_test_metrics, save_path, is_prediction_run=False):
    with open(save_path, 'w') as f:
        run_type = "Prediction" if is_prediction_run else "Training"
        f.write(f"===== {run_type} Run Summary =====\n")
        f.write(f"Timestamp: {config.get('RUN_TIMESTAMP_STR', 'N/A')}\n")
        f.write(f"Output Dir: {config.get('OUTPUT_SUBFOLDER_PATH', 'N/A')}\n\n")

        f.write("--- Hyperparameters ---\n")
        for k, v in final_params.items(): f.write(f"  {k}: {v}\n")

        if not is_prediction_run:
            f.write("\n--- Best Validation Epoch Performance ---\n")
            for k, v in best_val_metrics.items(): f.write(f"  {k.capitalize()}: {v:.4f}\n")

        f.write("\n--- Final Test Set Performance ---\n")
        f.write(f"Threshold Used: {final_test_metrics.get('threshold', 'N/A'):.4f}\n")
        for k, v in final_test_metrics.items():
            if k not in ['report', 'threshold']: f.write(f"  {k.capitalize()}: {v:.4f}\n")
        f.write("\nClassification Report:\n" + final_test_metrics.get('report', "N/A") + "\n")
    logging.info(f"Results summary saved to: {save_path}")

# ========================================================
# Main Execution Logic (Training Pipeline)
# ========================================================
def run_training_pipeline(data_df, config):
    pipeline_start_time = time.time()
    logging.info(f"--- Starting Transformer Training Pipeline (v{config.get('VERSION', 'unknown')}) ---")
    set_seed(config['SEED'])

    timestamp_str = datetime.now().strftime("run_%Y%m%d_%H%M%S")
    output_subfolder_path = os.path.join(config['BASE_OUTPUT_DIR'], timestamp_str)
    os.makedirs(output_subfolder_path, exist_ok=True)
    run_config = {**config, 'OUTPUT_SUBFOLDER_PATH': output_subfolder_path, 'RUN_TIMESTAMP_STR': timestamp_str}

    X_train, y_train, X_val, y_val, X_test, y_test, n_features, _, y_train_raw, test_indices = split_apply_undersample_scale(data_df, run_config)
    train_dataset = TimeSeriesDataset(X_train, y_train)
    val_dataset = TimeSeriesDataset(X_val, y_val)
    test_dataset = TimeSeriesDataset(X_test, y_test) if X_test.size > 0 else None

    final_params = {k.replace('FIXED_', '').lower(): v for k,v in run_config.items() if k.startswith('FIXED_')}
    model_init_params = {'n_features': n_features, **{k:v for k,v in final_params.items() if k in TransformerClassifier.__init__.__code__.co_varnames}}

    device = run_config['DEVICE']
    final_model = TransformerClassifier(**model_init_params).to(device)
    final_optimizer = optim.AdamW(final_model.parameters(), lr=final_params['lr'], weight_decay=final_params['weight_decay'])
    final_criterion = nn.BCEWithLogitsLoss()

    history = {k: [] for k in ['train_loss', 'val_loss', 'val_f1', 'val_acc', 'val_auc', 'val_gmean', 'avg_grad_norm']}
    best_val_metric, best_epoch_num, epochs_no_improve, best_val_metrics = -float('inf'), 0, 0, {}
    metric_to_monitor = run_config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'gmean')

    live_fig = None
    if ipython_display_available and run_config.get('VERBOSE_PLOT', False):
        try:
            live_fig = go.FigureWidget(make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Loss", "Metrics", "Grad Norm")))
            traces = [('Train Loss', 1, 1), ('Val Loss', 1, 1), ('Val F1', 2, 1), ('Val Acc', 2, 1), ('Val AUC', 2, 1), ('Val G-mean', 2, 1), ('Avg Grad Norm', 3, 1)]
            for name, r, c in traces: live_fig.add_trace(go.Scatter(name=name, mode='lines+markers'), row=r, col=c)
            live_fig.update_layout(title=f'Live Training (Monitoring: {metric_to_monitor.upper()})', height=800)
            display(live_fig)
            logging.info("Live plot initialized.")
        except Exception as e:
            logging.error(f"Failed to initialize live plot: {e}"); live_fig = None

    for epoch in range(run_config['FINAL_N_EPOCHS']):
        train_loss, avg_grad_norm = train_epoch(final_model, DataLoader(train_dataset, batch_size=final_params['batch_size'], shuffle=True), final_criterion, final_optimizer, device, run_config['GRADIENT_CLIP_MAX_NORM'])
        val_metrics, y_true_val, y_prob_val = evaluate(final_model, DataLoader(val_dataset, batch_size=final_params['batch_size']), final_criterion, device, return_preds=True)

        history['train_loss'].append(train_loss); history['avg_grad_norm'].append(avg_grad_norm)
        for k,v in val_metrics.items(): history[f'val_{k}'].append(v)

        logging.info(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val {metric_to_monitor.upper()}: {val_metrics[metric_to_monitor]:.4f}")

        if live_fig:
            with live_fig.batch_update():
                for i, key in enumerate(history.keys()): live_fig.data[i].x, live_fig.data[i].y = list(range(1, epoch + 2)), history[key]

        if epoch + 1 >= run_config['MIN_EPOCH_FINAL'] and val_metrics[metric_to_monitor] > best_val_metric:
            best_val_metric, best_epoch_num, epochs_no_improve, best_val_metrics = val_metrics[metric_to_monitor], epoch + 1, 0, val_metrics
            optimal_threshold = find_optimal_threshold(y_true_val, y_prob_val, run_config['THRESHOLD_OPTIMIZATION_METRIC'])
            torch.save({'model_state_dict': final_model.state_dict(), 'model_init_params': model_init_params, 'optimal_threshold': optimal_threshold}, os.path.join(output_subfolder_path, run_config['BEST_MODEL_FILENAME']))
            logging.info(f"  => New best model saved at epoch {best_epoch_num} with {metric_to_monitor.upper()} of {best_val_metric:.4f}")
        elif epoch + 1 >= run_config['MIN_EPOCH_FINAL']:
            epochs_no_improve += 1
            if epochs_no_improve >= run_config['FINAL_EARLY_STOPPING_PATIENCE']: logging.info("Early stopping triggered."); break

    save_static_training_plot(history, best_epoch_num, metric_to_monitor, os.path.join(output_subfolder_path, run_config['TRAINING_HISTORY_PLOT_FILENAME']))

    logging.info("\n--- Final Evaluation on TEST Set ---")
    model_path = os.path.join(output_subfolder_path, run_config['BEST_MODEL_FILENAME'])
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        eval_model = TransformerClassifier(**checkpoint['model_init_params']).to(device)
        eval_model.load_state_dict(checkpoint['model_state_dict'])
        test_metrics, y_true_test, y_prob_test = evaluate(eval_model, DataLoader(test_dataset, batch_size=final_params['batch_size']), None, device, return_preds=True)

        optimal_threshold = checkpoint['optimal_threshold']
        y_pred_test = (y_prob_test >= optimal_threshold).astype(int)
        final_test_metrics = {**test_metrics, 'threshold': optimal_threshold, 'report': classification_report(y_true_test, y_pred_test, digits=4, zero_division=0)}

        pred_df = pd.DataFrame({'actual': y_true_test, 'predicted_prob': y_prob_test, 'predicted_class': y_pred_test}, index=test_indices)
        pred_df.to_csv(os.path.join(output_subfolder_path, run_config['PREDICTION_DATAFRAME_FILENAME']))
        plot_confusion_matrix(y_true_test, y_pred_test, os.path.join(output_subfolder_path, run_config['CONFUSION_MATRIX_FILENAME']), f" (Test, Thresh={optimal_threshold:.2f})")
        plot_actual_vs_prediction(pred_df, os.path.join(output_subfolder_path, run_config['ACTUAL_VS_PREDICTION_PLOT_FILENAME']), optimal_threshold)

        save_results_summary(run_config, final_params, best_val_metrics, final_test_metrics, os.path.join(output_subfolder_path, run_config['RESULTS_SUMMARY_FILENAME']))

    logging.info(f"--- Training Pipeline Finished in {time.time() - pipeline_start_time:.2f}s ---")
    return output_subfolder_path

# ========================================================
# Prediction Pipeline Function
# ========================================================
def run_prediction_pipeline(data_df, model_path, base_config):
    pipeline_start_time = time.time()
    logging.info(f"\n--- Starting Prediction Pipeline (Model: {model_path}) ---")
    if not os.path.exists(model_path): logging.critical("Model file not found."); return

    device = base_config['DEVICE']
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model_init_params = checkpoint['model_init_params']
    optimal_threshold = checkpoint['optimal_threshold']

    timestamp_str = datetime.now().strftime("prediction_%Y%m%d_%H%M%S")
    pred_output_path = os.path.join(base_config['BASE_OUTPUT_DIR'], timestamp_str)
    os.makedirs(pred_output_path, exist_ok=True)
    pred_run_config = {**base_config, 'OUTPUT_SUBFOLDER_PATH': pred_output_path, 'RUN_TIMESTAMP_STR': timestamp_str, 'LOAD_MODEL_PATH': model_path, 'loaded_model_params': model_init_params}

    _, _, _, _, X_test, y_test, _, _, _, test_indices = split_apply_undersample_scale(data_df, pred_run_config)
    if X_test.size == 0: logging.warning("Test set empty, skipping prediction."); return
    test_dataset = TimeSeriesDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=base_config['DEFAULT_BATCH_SIZE'], shuffle=False)

    model = TransformerClassifier(**model_init_params).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    test_metrics, y_true_test, y_prob_test = evaluate(model, test_loader, None, device, return_preds=True)

    y_pred_test = (y_prob_test >= optimal_threshold).astype(int)
    final_test_metrics = {**test_metrics, 'threshold': optimal_threshold, 'report': classification_report(y_true_test, y_pred_test, digits=4, zero_division=0)}

    if test_indices is not None and len(test_indices) == len(y_true_test):
        pred_df = pd.DataFrame({'actual': y_true_test, 'predicted_prob': y_prob_test, 'predicted_class': y_pred_test}, index=test_indices)
        pred_df.to_csv(os.path.join(pred_output_path, pred_run_config['PREDICTION_DATAFRAME_FILENAME']))
        plot_confusion_matrix(y_true_test, y_pred_test, os.path.join(pred_output_path, pred_run_config['CONFUSION_MATRIX_FILENAME']), f" (Prediction, Thresh={optimal_threshold:.2f})")
        plot_actual_vs_prediction(pred_df, os.path.join(pred_output_path, pred_run_config['ACTUAL_VS_PREDICTION_PLOT_FILENAME']), optimal_threshold)

    save_results_summary(pred_run_config, model_init_params, None, final_test_metrics, os.path.join(pred_output_path, pred_run_config['RESULTS_SUMMARY_FILENAME']), is_prediction_run=True)
    logging.info(f"--- Prediction Pipeline Finished in {time.time() - pipeline_start_time:.2f}s. Results in {pred_output_path} ---")


# ========================================================
# Standalone Execution Block
# ========================================================
if __name__ == "__main__":

    config_run = {
        # --- General & Logging ---
        'SEED': 42, 'DEVICE_STR': "cuda" if torch.cuda.is_available() else "cpu",
        'VERBOSE_PLOT': True, # Set to True for live plotting in notebooks
        'VERBOSE_LOG': True,  # Set to True for detailed console logs

        # --- Data & Preprocessing ---
        'SEQUENCE_LENGTH': 10, 'TRAIN_SPLIT_RATIO': 0.7, 'VALIDATION_SPLIT_RATIO': 0.15,

        # --- Model Architecture ---
        'SKIP_OPTUNA_AND_USE_FIXED_PARAMS': True,
        'FIXED_D_MODEL': 32, 'FIXED_N_HEADS': 4, 'FIXED_N_ENCODER_LAYERS': 2, 'FIXED_DIM_FEEDFORWARD': 64,
        'FIXED_TRANSFORMER_DROPOUT': 0.1, 'FIXED_FC_DROPOUT': 0.15, 'FIXED_LR': 1e-3, 'FIXED_BATCH_SIZE': 256, 'FIXED_WEIGHT_DECAY': 1e-4,

        # --- Training & Optimization ---
        'OPTIMIZE_THRESHOLD': True, 'THRESHOLD_OPTIMIZATION_METRIC': 'gmean', 'TUNE_HYPERPARAMETERS': False,
        'GRADIENT_CLIP_MAX_NORM': 1.0, 'FINAL_N_EPOCHS': 100, 'FINAL_EARLY_STOPPING_PATIENCE': 15, 'MIN_EPOCH_FINAL': 10,

        # --- Optuna ---
        'OPTUNA_METRIC_TO_OPTIMIZE': "gmean",

        # --- Output Filenames ---
        'BASE_OUTPUT_DIR': 'models_transformer_final',
        'BEST_MODEL_FILENAME': 'best_transformer_model.pth',
        'TRAINING_HISTORY_PLOT_FILENAME': "training_history.png",
        'CONFUSION_MATRIX_FILENAME': "confusion_matrix.png",
        'RESULTS_SUMMARY_FILENAME': "results_summary.txt",
        'PREDICTION_DATAFRAME_FILENAME': "test_predictions.csv",
        'ACTUAL_VS_PREDICTION_PLOT_FILENAME': "actual_vs_prediction_plot.png",
        'VERSION': '2.3.0_transformer_final'
    }
    config_run['DEVICE'] = torch.device(config_run['DEVICE_STR'])
    setup_logging(config_run['VERBOSE_LOG'])

    try:
        logging.info("Creating dummy data for demonstration...")
        num_samples, num_features = 2000, 10
        dates = pd.to_datetime(pd.date_range(start='2022-01-01', periods=num_samples, freq='H'))
        features = np.random.randn(num_samples, num_features)
        signal = pd.Series(features[:, 0]).rolling(window=10).mean().fillna(0)
        noise = np.random.rand(num_samples) * 0.5
        target = (signal + noise > np.median(signal + noise)).astype(int)
        data_df = pd.DataFrame(features, index=dates, columns=[f'feature_{i}' for i in range(num_features)])
        data_df['target'] = target

        training_output_dir = run_training_pipeline(data_df, config_run)

        if training_output_dir:
            model_to_load_path = os.path.join(training_output_dir, config_run['BEST_MODEL_FILENAME'])
            if os.path.exists(model_to_load_path):
                run_prediction_pipeline(data_df, model_to_load_path, config_run)
    except Exception as main_exec_e:
        logging.critical(f"An error occurred in the main execution block: {main_exec_e}", exc_info=True)


In [ ]:
# set_sequence_classifier.py

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. Configuration Dictionary
# ==============================================================================
# Centralized configuration for all hyperparameters and settings.
config = {
    # --- Data Configuration ---
    "data": {
        "test_size": 0.2,
        "sequence_length": 50,
        "target_col": "target",
        "unit_id_col": "unit_id",
        "time_col": "time_period"
    },
    # --- Model Architecture ---
    "model": {
        "d_model": 128,
        "d_set_summary": 16,
        "phi_hidden_dim": 256,
        "rho_hidden_dim": 64,
        "psi_hidden_dim": 128,
        "n_set_seq_layers": 4,
        "n_seq_layers": 2,
        "dropout": 0.1,
        "long_conv_kernel_size": 32,
    },
    # --- Training Configuration ---
    "training": {
        "loss_function": "gmean", # Options: "bce", "gmean"
        "learning_rate": 0.001,
        "batch_size": 32, # Reduced batch size to accommodate larger tensors
        "n_epochs": 20,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "scaler": "standard",
    },
    # --- Cross-Validation ---
    "cross_val": {
        "n_splits": 5
    }
}

# ==============================================================================
# 2. Model Components & Architecture
# ==============================================================================

class GMeanLoss(nn.Module):
    """
    A differentiable loss function to maximize the G-mean.
    Loss = 1 - G-mean = 1 - sqrt(Sensitivity * Specificity).
    """
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, labels):
        preds = torch.sigmoid(logits)
        labels = labels.float()

        tp = torch.sum(preds * labels)
        sensitivity = tp / (torch.sum(labels) + self.epsilon)
        specificity = torch.sum((1 - preds) * (1 - labels)) / (torch.sum(1 - labels) + self.epsilon)

        g_mean = torch.sqrt(sensitivity * specificity + self.epsilon)
        return 1 - g_mean

class LongConv(nn.Module):
    """A simple 1D causal convolution layer."""
    def __init__(self, d_model, kernel_size, dropout):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
            groups=d_model
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x[:, :, :-(self.conv.kernel_size[0] - 1)]
        x = x.transpose(1, 2)
        return self.dropout(x)


class SetSequenceLayer(nn.Module):
    """Implements a single Set-Sequence layer."""
    def __init__(self, config):
        super().__init__()
        d_model = config["model"]["d_model"]
        d_set_summary = config["model"]["d_set_summary"]
        phi_hidden = config["model"]["phi_hidden_dim"]
        rho_hidden = config["model"]["rho_hidden_dim"]
        psi_hidden = config["model"]["psi_hidden_dim"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.phi = nn.Sequential(nn.Linear(d_model, phi_hidden), nn.ReLU(), nn.Linear(phi_hidden, d_model))
        self.rho = nn.Sequential(nn.Linear(d_model, rho_hidden), nn.ReLU(), nn.Linear(rho_hidden, d_set_summary))
        self.psi = nn.Sequential(nn.Linear(d_model + d_set_summary, psi_hidden), nn.ReLU(), nn.Linear(psi_hidden, d_model))
        self.seq_layer = LongConv(d_model, kernel_size, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, num_units, seq_len, d_model = x.shape

        x_reshaped = x.view(batch_size * num_units, seq_len, d_model)
        phi_x = self.phi(x_reshaped).view(batch_size, num_units, seq_len, d_model)
        mean_phi_x = torch.mean(phi_x, dim=1)
        set_summary = self.rho(mean_phi_x)

        set_summary_expanded = set_summary.unsqueeze(1).expand(-1, num_units, -1, -1)
        augmented_x = torch.cat([x, set_summary_expanded], dim=-1)
        augmented_x_reshaped = augmented_x.view(batch_size * num_units, seq_len, -1)
        psi_out = self.psi(augmented_x_reshaped)

        res_x = x.view(batch_size * num_units, seq_len, d_model)
        processed_x = self.norm1(res_x + psi_out)
        seq_out = self.seq_layer(processed_x)
        final_out = self.norm2(processed_x + seq_out)

        return final_out.view(batch_size, num_units, seq_len, d_model)


class SetSequenceClassifier(nn.Module):
    """The full Set-Sequence model for per-unit classification."""
    def __init__(self, config, n_features):
        super().__init__()
        d_model = config["model"]["d_model"]
        n_set_seq_layers = config["model"]["n_set_seq_layers"]
        n_seq_layers = config["model"]["n_seq_layers"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.input_projection = nn.Linear(n_features, d_model)
        self.set_seq_layers = nn.ModuleList([SetSequenceLayer(config) for _ in range(n_set_seq_layers)])
        self.final_seq_layers = nn.ModuleList([LongConv(d_model, kernel_size, dropout) for _ in range(n_seq_layers)])
        self.classifier_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input of shape (batch_size, num_units, seq_len, n_features)
        Returns:
            torch.Tensor: Logits of shape (batch_size, num_units)
        """
        batch_size, num_units, seq_len, _ = x.shape

        # 1. Project input features to d_model
        x = self.input_projection(x)

        # 2. Pass through Set-Sequence layers
        for layer in self.set_seq_layers:
            x = layer(x)

        # 3. Reshape to process all units' sequences in one go
        # (batch * units, seq_len, d_model)
        x = x.view(batch_size * num_units, seq_len, -1)

        # 4. Pass through final sequence-only layers
        for layer in self.final_seq_layers:
            x = x + layer(x)

        # 5. Use the last time step's output for classification
        last_time_step = x[:, -1, :]

        # 6. Get logits from the classifier head
        logits = self.classifier_head(last_time_step)

        # 7. Reshape back to (batch_size, num_units)
        return logits.view(batch_size, num_units)

# ==============================================================================
# 3. Data Preparation and Utilities
# ==============================================================================

def create_sequences(df, config):
    """
    Transforms the DataFrame into sequences and per-unit targets.
    X shape: (num_sequences, num_units, seq_len, n_features)
    y shape: (num_sequences, num_units)
    """
    seq_len = config["data"]["sequence_length"]
    unit_col = config["data"]["unit_id_col"]
    time_col = config["data"]["time_col"]
    target_col = config["data"]["target_col"]

    feature_cols = [c for c in df.columns if c not in [unit_col, time_col, target_col]]
    n_features = len(feature_cols)

    df = df.sort_values(by=[time_col, unit_col])

    sequences, targets = [], []
    unique_times = df[time_col].unique()

    for t in range(seq_len, len(unique_times)):
        start_time, end_time, target_time = unique_times[t - seq_len], unique_times[t - 1], unique_times[t]

        sequence_df = df[(df[time_col] >= start_time) & (df[time_col] <= end_time)]
        target_df = df[df[time_col] == target_time]

        # Get the units present in this time window
        units_in_window = sequence_df[unit_col].unique()

        # Pivot features
        seq_pivot = sequence_df.pivot(index=unit_col, columns=time_col, values=feature_cols).fillna(0)

        # Pivot targets
        target_pivot = target_df.pivot(index=unit_col, columns=time_col, values=target_col)

        # Align units between features and targets, fill missing targets with 0
        seq_pivot, target_pivot = seq_pivot.align(target_pivot, join='left', axis=0, fill_value=0)

        num_units = len(seq_pivot)
        if num_units == 0: continue

        try:
            seq_array = seq_pivot.values.reshape(num_units, seq_len, n_features)
            target_array = target_pivot.values.flatten()

            sequences.append(seq_array)
            targets.append(target_array)
        except ValueError:
            continue

    # Note: Sequences can have different numbers of units. This requires custom padding/batching.
    # For simplicity here, we filter for sequences with the most common number of units.
    if not sequences: return np.array([]), np.array([])

    unit_counts = [s.shape[0] for s in sequences]
    if not unit_counts: return np.array([]), np.array([])

    most_common_n_units = max(set(unit_counts), key=unit_counts.count)

    X_filtered = [sequences[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]
    y_filtered = [targets[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]

    if not X_filtered: return np.array([]), np.array([])

    return np.stack(X_filtered), np.stack(y_filtered)


def get_scaler(name):
    if name == "standard": return StandardScaler()
    if name == "minmax": return MinMaxScaler()
    return None

def get_criterion(name):
    if name == "bce": return nn.BCEWithLogitsLoss()
    if name == "gmean": return GMeanLoss()
    raise ValueError(f"Unknown loss function: {name}")

def calculate_gmean(labels, preds, epsilon=1e-8):
    if len(np.unique(labels)) < 2: return 0.0
    cm = confusion_matrix(labels, np.round(preds))
    if cm.shape != (2, 2): return 0.0
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn + epsilon)
    specificity = tn / (tn + fp + epsilon)
    return np.sqrt(sensitivity * specificity)

# ==============================================================================
# 4. Training and Evaluation Loop
# ==============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch) # Shape: (batch, units)

        # Flatten outputs and targets for loss calculation
        loss = criterion(outputs.view(-1), y_batch.view(-1).float())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            # Flatten for loss and metrics
            flat_outputs = outputs.view(-1)
            flat_labels = y_batch.view(-1).float()

            loss = criterion(flat_outputs, flat_labels)
            total_loss += loss.item()

            all_preds.extend(torch.sigmoid(flat_outputs).cpu().numpy())
            all_labels.extend(flat_labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, np.round(all_preds))
    gmean = calculate_gmean(all_labels, all_preds)

    return avg_loss, auc, acc, gmean

# ==============================================================================
# 5. Main Execution
# ==============================================================================

def main():
    print("--- Set-Sequence Model for Per-Unit Classification ---")
    print(f"Using device: {config['training']['device']}")
    print(f"Optimizing with loss function: {config['training']['loss_function'].upper()}")

    print("Generating sample data...")
    n_units, n_time_periods, n_features = 50, 500, 10
    data = []
    for unit in range(n_units):
        for time in range(n_time_periods):
            row = {'unit_id': unit, 'time_period': time}
            features = np.sin(time / 50 + unit) + np.random.randn(n_features) * 0.5
            for i, f_val in enumerate(features): row[f'feature_{i}'] = f_val
            row['target'] = 1 if (features[0] + np.sin(time/20)) > 0.8 else 0
            data.append(row)
    df = pd.DataFrame(data)
    print(f"Sample data created. Target distribution:\n{df['target'].value_counts(normalize=True)}")

    feature_cols = [c for c in df.columns if isinstance(c, str) and c.startswith('feature')]
    time_col = config['data']['time_col']
    test_split_time = df[time_col].unique()[int(len(df[time_col].unique()) * (1 - config['data']['test_size']))]
    df_train_val, df_test = df[df[time_col] < test_split_time], df[df[time_col] >= test_split_time]

    scaler = get_scaler(config['training']['scaler'])
    if scaler:
        print(f"Applying {config['training']['scaler']} scaling...")
        df_train_val.loc[:, feature_cols] = scaler.fit_transform(df_train_val[feature_cols])
        df_test.loc[:, feature_cols] = scaler.transform(df_test[feature_cols])

    print("\nStarting walk-forward cross-validation...")
    tscv = TimeSeriesSplit(n_splits=config['cross_val']['n_splits'])
    fold_results = []
    time_periods = df_train_val[time_col].unique()

    for fold, (train_indices, val_indices) in enumerate(tscv.split(time_periods)):
        print(f"\n--- Fold {fold + 1}/{config['cross_val']['n_splits']} ---")
        df_train_fold = df_train_val[df_train_val[time_col].isin(time_periods[train_indices])]
        df_val_fold = df_train_val[df_train_val[time_col].isin(time_periods[val_indices])]

        X_train, y_train = create_sequences(df_train_fold, config)
        X_val, y_val = create_sequences(df_val_fold, config)

        if X_train.shape[0] == 0 or X_val.shape[0] == 0:
            print("Skipping fold due to insufficient data to create sequences.")
            continue

        train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()), batch_size=config['training']['batch_size'], shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).long()), batch_size=config['training']['batch_size'])

        model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
        optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
        criterion = get_criterion(config['training']['loss_function'])

        for epoch in range(config['training']['n_epochs']):
            train_loss = train_epoch(model, train_loader, optimizer, criterion, config['training']['device'])
            val_loss, val_auc, val_acc, val_gmean = evaluate(model, val_loader, criterion, config['training']['device'])
            if (epoch + 1) % 5 == 0:
                 print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val G-mean: {val_gmean:.4f}")

        fold_results.append({'auc': val_auc, 'acc': val_acc, 'gmean': val_gmean})

    print("\n--- Final Evaluation on Test Set ---")
    print("Retraining model on full train/val data...")
    X_train_full, y_train_full = create_sequences(df_train_val, config)
    X_test, y_test = create_sequences(df_test, config)

    if X_train_full.shape[0] == 0 or X_test.shape[0] == 0:
        print("Cannot perform final evaluation due to insufficient data.")
        return

    train_full_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_full).float(), torch.from_numpy(y_train_full).long()), batch_size=config['training']['batch_size'], shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long()), batch_size=config['training']['batch_size'])

    final_model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
    optimizer = optim.Adam(final_model.parameters(), lr=config['training']['learning_rate'])
    criterion = get_criterion(config['training']['loss_function'])

    for epoch in range(config['training']['n_epochs']):
        train_loss = train_epoch(final_model, train_full_loader, optimizer, criterion, config['training']['device'])
        if (epoch + 1) % 5 == 0: print(f"Retraining Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}")

    test_loss, test_auc, test_acc, test_gmean = evaluate(final_model, test_loader, criterion, config['training']['device'])

    print("\n--- Results Summary ---")
    if fold_results:
        avg_cv_gmean = np.mean([r['gmean'] for r in fold_results])
        print(f"Average Cross-Validation G-mean: {avg_cv_gmean:.4f}")

    print(f"\nFinal Test Set G-mean: {test_gmean:.4f}")
    print(f"Final Test Set AUC: {test_auc:.4f}")
    print(f"Final Test Set Accuracy: {test_acc:.4f}")


if __name__ == "__main__":
    main()
